# 01 — Data Profiling: Indonesian Political News

**Milestone 1 | Analisis Big Data**

Dataset: [ardimardiana/indonesian-political-news-clean](https://huggingface.co/datasets/ardimardiana/indonesian-political-news-clean). Unit analisis adalah **artikel berita**, bukan individu atau seluruh populasi media Indonesia.

Notebook ini hanya memeriksa data, **tidak menghapus atau mengubah file mentah**. Semua angka wajib berasal dari eksekusi dataset lokal. Jalankan notebook dari atas ke bawah.

## 1. Lingkungan kerja

Jalankan instalasi berikut jika berada di Google Colab atau lingkungan Python yang belum menyiapkan dependencies. Di Docker, instalasi sudah dilakukan oleh `requirements.txt`. Bila memakai Google Drive di Colab, mount terlebih dahulu melalui `from google.colab import drive; drive.mount('/content/drive')` dan pastikan path dataset sesuai.

In [14]:
# Uncomment hanya jika library belum tersedia:
# %pip install -r ../requirements.txt

## 2. Deteksi folder dan file dataset

In [2]:
from pathlib import Path
import sys
import polars as pl

# Isi manual hanya jika lokasi data berbeda dari struktur repository.
# Contoh Windows: r"C:\Users\NamaUser\bigdata_teman\data\raw\indonesian-political-news-clean"
# Contoh Colab setelah mount Drive: "/content/drive/MyDrive/bigdata_teman/data/raw/indonesian-political-news-clean"
DATASET_DIR_OVERRIDE = None

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

possible_dirs = [
    ROOT / "data" / "raw" / "indonesian-political-news-clean",
    Path.home() / "bigdata_teman" / "data" / "raw" / "indonesian-political-news-clean",
    Path("/content/drive/MyDrive/bigdata_teman/data/raw/indonesian-political-news-clean"),
]

if DATASET_DIR_OVERRIDE:
    DATA_DIR = Path(DATASET_DIR_OVERRIDE)
else:
    DATA_DIR = next((p for p in possible_dirs if p.exists()), possible_dirs[0])

PARQUET_FILES = sorted(DATA_DIR.rglob("*.parquet")) if DATA_DIR.exists() else []
if not PARQUET_FILES:
    raise FileNotFoundError(
        f"File Parquet tidak ditemukan di {DATA_DIR}. "
        "Sesuaikan DATASET_DIR_OVERRIDE pada cell ini atau ikuti data/README.md."
    )
PARQUET_PATHS = [str(p) for p in PARQUET_FILES]
print("Root repo:", ROOT)
print("Folder dataset:", DATA_DIR)
print("Jumlah file Parquet:", len(PARQUET_FILES))
print("Ukuran:", round(sum(f.stat().st_size for f in PARQUET_FILES)/1e9,3), "GB")
print("Versi Polars:", pl.__version__)

Root repo: C:\Users\raihan bhakti\bigdata_teman
Folder dataset: C:\Users\raihan bhakti\bigdata_teman\data\raw\indonesian-political-news-clean
Jumlah file Parquet: 22
Ukuran: 5.446 GB
Versi Polars: 1.44.2


## 3. Lazy scan dan schema

`scan_parquet` membuat LazyFrame, dan `collect_schema` membaca struktur tanpa memuat seluruh isi dataset ke RAM. Scan seluruh file `.parquet` dari lokasi yang ditemukan.

In [3]:
lf = pl.scan_parquet(PARQUET_PATHS)
schema = lf.collect_schema()
print("Jumlah kolom:", len(schema.names()))
print("Schema:")
for name, dtype in schema.items():
    print(f"  {name}: {dtype}")
print("Bentuk objek:", type(lf).__name__)

Jumlah kolom: 15
Schema:
  id: String
  news_title: String
  news_source: String
  news_author: String
  news_hostname: String
  news_date: String
  news_text: String
  news_tags: String
  news_kabkot: String
  news_intent: String
  news_image: String
  update_date: String
  create_date: String
  news_guid: String
  news_raw_data: String
Bentuk objek: LazyFrame


## 4. Ukuran dan total baris

In [4]:
total_rows = lf.select(pl.len().alias("total_baris")).collect().item()
total_bytes = sum(f.stat().st_size for f in PARQUET_FILES)
print("Total baris aktual:", f"{total_rows:,}")
print("Jumlah file:", len(PARQUET_FILES))
print("Ukuran GB:", round(total_bytes / 1_000_000_000, 3))
print("Ukuran GiB:", round(total_bytes / (1024**3), 3))
if total_rows <= 1_000_000 and total_bytes < 500_000_000:
    print("PERINGATAN: batas minimal dataset tugas belum terpenuhi.")
else:
    print("Ukuran/jumlah baris memenuhi salah satu ambang dataset tugas.")

Total baris aktual: 1,628,288
Jumlah file: 22
Ukuran GB: 5.446
Ukuran GiB: 5.072
Ukuran/jumlah baris memenuhi salah satu ambang dataset tugas.


## 5. Sampel dan struktur kolom penting

In [12]:
wanted = ["id", "news_title", "news_hostname", "news_date", "news_kabkot", "news_text"]
missing = [col for col in wanted if col not in schema]
assert not missing, f"Kolom wajib belum ditemukan: {missing}"
# Hanya menampilkan ringkasan teks agar contoh tidak terlalu panjang.
lf.select([
    "id", "news_title", "news_hostname", "news_date", "news_kabkot",
    pl.col("news_text").cast(pl.String).str.slice(0, 120).alias("cuplikan_teks")
]).head(5).collect()

id,news_title,news_hostname,news_date,news_kabkot,cuplikan_teks
str,str,str,str,str,str
"""1924973""","""Dana Kegiatan Isbat Nikah Baka…","""disway.id""","""2026-04-27 01:30:00+00:00""","""Kabupaten Mukomuko""","""Dana Kegiatan Isbat Nikah Baka…"
"""1925951""","""Satu Orang Meninggal dari Insi…","""ayokaltim.com""","""2026-04-27 03:19:46+00:00""","""Kota Samarinda""","""Satu Orang Meninggal dari Insi…"
"""1925952""","""DPRD Bongkar Ketidaksesuaian L…","""infosatu.co""","""2026-04-27 03:59:29+00:00""","""Kota Samarinda""","""Samarinda, infosatu.co – Klaim…"
"""1925953""","""DPRD Dorong Penanganan Banjir …","""infosatu.co""","""2026-04-27 01:43:24+00:00""","""Kota Samarinda""","""Samarinda, infosatu.co – Dewan…"
"""1924974""","""Kepung Pasar Karang Nabire, Ma…","""tribunnews.com""","""2026-04-27 02:14:39+00:00""","""Kabupaten Nabire""","""Demo di Nabire Kepung Pasar Ka…"


## 6. Profiling missing value dan string kosong

In [5]:
# Null pada semua kolom, termasuk kolom nested yang tidak dipakai di EDA.
null_counts = lf.select(pl.all().null_count()).collect()
print(null_counts)

focus = ["news_title", "news_text", "news_hostname", "news_kabkot", "news_date"]
blank_counts = lf.select([
    (pl.col(c).cast(pl.String).fill_null("").str.strip_chars() == "").sum().alias(c)
    for c in focus
]).collect()
print("Null ATAU string kosong pada kolom penting:")
print(blank_counts)

shape: (1, 15)
┌─────┬────────────┬─────────────┬─────────────┬───┬─────────────┬─────────────┬───────────┬───────────────┐
│ id  ┆ news_title ┆ news_source ┆ news_author ┆ … ┆ update_date ┆ create_date ┆ news_guid ┆ news_raw_data │
│ --- ┆ ---        ┆ ---         ┆ ---         ┆   ┆ ---         ┆ ---         ┆ ---       ┆ ---           │
│ u32 ┆ u32        ┆ u32         ┆ u32         ┆   ┆ u32         ┆ u32         ┆ u32       ┆ u32           │
╞═════╪════════════╪═════════════╪═════════════╪═══╪═════════════╪═════════════╪═══════════╪═══════════════╡
│ 0   ┆ 0          ┆ 0           ┆ 149514      ┆ … ┆ 0           ┆ 1281020     ┆ 347268    ┆ 303240        │
└─────┴────────────┴─────────────┴─────────────┴───┴─────────────┴─────────────┴───────────┴───────────────┘
Null ATAU string kosong pada kolom penting:
shape: (1, 5)
┌────────────┬───────────┬───────────────┬─────────────┬───────────┐
│ news_title ┆ news_text ┆ news_hostname ┆ news_kabkot ┆ news_date │
│ ---        ┆ ---       ┆

## 7. Profiling tanggal

`news_date` memiliki contoh timestamp ISO dengan zona waktu. Untuk analisis berbasis hari, sepuluh karakter awal diambil sebagai `YYYY-MM-DD`, lalu diparse menjadi `Date`. Ini adalah **tanggal kalender yang tercantum**, bukan konversi otomatis antar-zona waktu.

In [8]:
date_lf = lf.select(
    pl.col("news_date").cast(pl.String).str.slice(0, 10)
      .str.strptime(pl.Date, "%Y-%m-%d", strict=False).alias("tanggal")
)
print(date_lf.select([
    pl.col("tanggal").min().alias("tanggal_min"),
    pl.col("tanggal").max().alias("tanggal_max"),
    pl.col("tanggal").null_count().alias("tanggal_gagal_parse"),
]).collect())

shape: (1, 3)
┌─────────────┬─────────────┬─────────────────────┐
│ tanggal_min ┆ tanggal_max ┆ tanggal_gagal_parse │
│ ---         ┆ ---         ┆ ---                 │
│ date        ┆ date        ┆ u32                 │
╞═════════════╪═════════════╪═════════════════════╡
│ 0001-11-29  ┆ 2569-03-20  ┆ 84332               │
└─────────────┴─────────────┴─────────────────────┘


In [10]:
import polars as pl

# Mengambil bagian tanggal dengan format YYYY-MM-DD
tanggal_cek = lf.with_columns(
    pl.col("news_date")
    .str.strip_chars()
    .str.extract(r"^(\d{4}-\d{2}-\d{2})", 1)
    .str.strptime(
        pl.Date,
        format="%Y-%m-%d",
        strict=False
    )
    .alias("tanggal_parsed")
)

# Mendeteksi tanggal di luar rentang pemeriksaan
anomali_tanggal = tanggal_cek.filter(
    pl.col("tanggal_parsed").is_not_null()
    &
    (
        (pl.col("tanggal_parsed").dt.year() < 2000)
        |
        (pl.col("tanggal_parsed").dt.year() > 2026)
    )
)

print("Jumlah kandidat anomali tanggal:")

print(
    anomali_tanggal
    .select(pl.len().alias("jumlah_anomali"))
    .collect()
)

print("Contoh tanggal tidak wajar:")

print(
    anomali_tanggal
    .select([
        "news_date",
        "tanggal_parsed",
        "news_title"
    ])
    .head(10)
    .collect()
)

Jumlah kandidat anomali tanggal:
shape: (1, 1)
┌────────────────┐
│ jumlah_anomali │
│ ---            │
│ u32            │
╞════════════════╡
│ 403            │
└────────────────┘
Contoh tanggal tidak wajar:
shape: (10, 3)
┌───────────────────────────┬────────────────┬─────────────────────────────────┐
│ news_date                 ┆ tanggal_parsed ┆ news_title                      │
│ ---                       ┆ ---            ┆ ---                             │
│ str                       ┆ date           ┆ str                             │
╞═══════════════════════════╪════════════════╪═════════════════════════════════╡
│ 2031-04-27 13:26:00+00:00 ┆ 2031-04-27     ┆ Peringati 30 Tahun Otonomi Dae… │
│ 2040-04-27 08:56:00+00:00 ┆ 2040-04-27     ┆ Semarang Membara! Black Dragbi… │
│ 2027-04-03 13:26:00+00:00 ┆ 2027-04-03     ┆ Dolanan Bareng Bunda PAUD, Cer… │
│ 2027-04-03 13:26:00+00:00 ┆ 2027-04-03     ┆ TP PKK Kota Salatiga Gaungkan … │
│ 2027-04-03 13:26:00+00:00 ┆ 2027-04-03     ┆ B

## 8. Distribusi sumber berita dan wilayah

In [13]:
for col in ["news_hostname", "news_kabkot"]:
    print("10 kategori terbanyak:", col)
    print(lf.group_by(col).agg(pl.len().alias("jumlah"))
            .sort("jumlah", descending=True).limit(10).collect())

10 kategori terbanyak: news_hostname
shape: (10, 2)
┌────────────────────┬────────┐
│ news_hostname      ┆ jumlah │
│ ---                ┆ ---    │
│ str                ┆ u32    │
╞════════════════════╪════════╡
│ tribunnews.com     ┆ 124593 │
│ rri.co.id          ┆ 93306  │
│ jawapos.com        ┆ 38184  │
│ pikiran-rakyat.com ┆ 37584  │
│ antaranews.com     ┆ 37217  │
│ disway.id          ┆ 30898  │
│ detik.com          ┆ 30644  │
│ kompas.com         ┆ 18587  │
│ merdeka.com        ┆ 13119  │
│ polri.go.id        ┆ 10028  │
└────────────────────┴────────┘
10 kategori terbanyak: news_kabkot
shape: (10, 2)
┌────────────────┬────────┐
│ news_kabkot    ┆ jumlah │
│ ---            ┆ ---    │
│ str            ┆ u32    │
╞════════════════╪════════╡
│ Kota Bandung   ┆ 16436  │
│ Kota Malang    ┆ 15576  │
│ Kota Medan     ┆ 14224  │
│ Kota Makassar  ┆ 13724  │
│ Kota Semarang  ┆ 13441  │
│ Kota Bogor     ┆ 13305  │
│ Kota Tangerang ┆ 12813  │
│ Kota Surabaya  ┆ 12720  │
│ Kota Padang    ┆ 113

## 9. Profiling panjang teks dan potensi duplikasi

Panjang teks luar biasa besar **ditandai sebagai anomali, bukan langsung dibuang**. Perkiraan duplikasi ditelusuri menggunakan kombinasi `news_hostname`, judul ternormalisasi, dan tanggal publikasi.

In [6]:
text_profile = lf.select([
    pl.col("news_title").cast(pl.String).str.len_chars().alias("panjang_judul"),
    pl.col("news_text").cast(pl.String).str.len_chars().alias("panjang_teks"),
]).select([
    pl.col("panjang_judul").median().alias("median_judul"),
    pl.col("panjang_teks").median().alias("median_teks"),
    pl.col("panjang_teks").quantile(0.99).alias("p99_teks"),
    pl.col("panjang_teks").max().alias("maks_teks"),
]).collect()
print(text_profile)

shape: (1, 4)
┌──────────────┬─────────────┬──────────┬───────────┐
│ median_judul ┆ median_teks ┆ p99_teks ┆ maks_teks │
│ ---          ┆ ---         ┆ ---      ┆ ---       │
│ f64          ┆ f64         ┆ f64      ┆ u32       │
╞══════════════╪═════════════╪══════════╪═══════════╡
│ 86.0         ┆ 1815.0      ┆ 5661.0   ┆ 1062327   │
└──────────────┴─────────────┴──────────┴───────────┘


In [7]:
kandidat_duplikat = (
    lf.select([
        pl.col("news_hostname").cast(pl.String).fill_null("Tidak diketahui").str.to_lowercase().alias("host"),
        pl.col("news_title").cast(pl.String).fill_null("").str.to_lowercase()
          .str.replace_all(r"\s+", " ").str.strip_chars().alias("judul_norm"),
        pl.col("news_date").cast(pl.String).str.slice(0, 10).alias("tanggal")
    ])
    .group_by(["host", "judul_norm", "tanggal"])
    .agg(pl.len().alias("frekuensi"))
    .filter(pl.col("frekuensi") > 1)
    .select((pl.col("frekuensi") - 1).sum().fill_null(0).alias("kandidat_duplikat"))
    .collect()
)
print(kandidat_duplikat)

shape: (1, 1)
┌───────────────────┐
│ kandidat_duplikat │
│ ---               │
│ u32               │
╞═══════════════════╡
│ 303               │
└───────────────────┘


## Ringkasan Milestone 1

Catat angka aktual dari hasil Run All pada bagian hasil profil di README. Jangan menyatakan data bebas duplikasi atau memiliki cakupan media nasional tanpa menguji representativitasnya. Temuan kualitas data diteruskan ke `02_data_cleaning.ipynb`.